# Stage 01 — Data Acquisition

**Owner:** Shuvechchha Pun (Data Acquisition & Context Lead)

**Objective.** Load the eight WGEA CSV files, inspect their schemas, run validity checks, load the optional external ABS heterogeneous source, and checkpoint the raw data dict for the downstream notebooks.

**Inputs.** `Dataset/wgea_public_dataset_2025/*.csv` (or the 5-row sample folder when `USE_SAMPLE=1`).

**Outputs.** `data/processed/checkpoints/01_raw_data.pkl` — dict of DataFrames keyed by short name.

**Why this stage matters.** Assessment 1 §2.2 described the dataset as a ZIP of 8 related CSVs keyed on `employer_abn`. Before any modelling we need to confirm every file is present, the join key is populated, and the reporting year is the intended 2024-25 cohort.

## 1. Setup

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src import config, data_acquisition
from src.utils import save_checkpoint

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

print("USE_SAMPLE    :", config.USE_SAMPLE)
print("Data source   :", config.ACTIVE_DATA_DIR)

USE_SAMPLE    : False
Data source   : D:\CDU\Semester3\PRT564_DATA_ANALYTICS_AND_VISUALISATION\project\Dataset\wgea_public_dataset_2025


## 2. Load the 8 WGEA CSVs

Each file contributes distinct information, so the dataset is intrinsically *heterogeneous in structure* (workforce counts vs questionnaire long tables) even before we add an external source. `load_wgea()` returns a dict keyed by short name.

In [2]:
data = data_acquisition.load_wgea()
summary = pd.DataFrame(
    [(name, df.shape[0], df.shape[1]) for name, df in data.items()],
    columns=["short_name", "rows", "cols"],
)
summary

21:39:40 | INFO    | src.data_acquisition | Loading WGEA CSVs from D:\CDU\Semester3\PRT564_DATA_ANALYTICS_AND_VISUALISATION\project\Dataset\wgea_public_dataset_2025
21:39:41 | INFO    | src.data_acquisition |   workforce_composition                         211659 rows   17 cols
21:39:41 | INFO    | src.data_acquisition |   workforce_management_statistics               235783 rows   18 cols
21:39:43 | INFO    | src.data_acquisition |   questionnaire_workplace_overview              233098 rows   20 cols
21:39:44 | INFO    | src.data_acquisition |   questionnaire_action_on_gender_equality       146020 rows   20 cols
21:39:46 | INFO    | src.data_acquisition |   questionnaire_employee_support                332122 rows   20 cols
21:39:47 | INFO    | src.data_acquisition |   questionnaire_flexible_work                   257127 rows   20 cols
21:39:51 | INFO    | src.data_acquisition |   questionnaire_harm_prevention                 630812 rows   20 cols
21:39:51 | INFO    | src.data_acquisi

,short_name,rows,cols
0,workforce_composition,211659,17
1,workforce_management_statistics,235783,18
2,questionnaire_workplace_overview,233098,20
3,questionnaire_action_on_gender_equality,146020,20
4,questionnaire_employee_support,332122,20
5,questionnaire_flexible_work,257127,20
6,questionnaire_harm_prevention,630812,20
7,questionnaire_catalogue,539,10


## 3. Inspect the two workforce tables

These are *wide-ish* tables where each row is one occupational/gender cell for an employer. They will be aggregated to one-row-per-employer in notebook 02.

In [3]:
print("workforce_composition columns:")
print(list(data["workforce_composition"].columns))
data["workforce_composition"].head()

workforce_composition columns:
['reporting_year', 'corporate_group_name', 'employer_name', 'employer_abn', 'is_relevant_employer', 'employer_size', 'anzsic_code', 'anzsic_division', 'anzsic_subdivision', 'anzsic_group', 'anzsic_class', 'manager_category', 'occupation', 'employment_status', 'employment_type', 'gender', 'n_employees']


,reporting_year,corporate_group_name,employer_name,employer_abn,is_relevant_employer,employer_size,anzsic_code,anzsic_division,anzsic_subdivision,anzsic_group,anzsic_class,manager_category,occupation,employment_status,employment_type,gender,n_employees
0,2024-25,1-STOP CONNECTIONS PTY LIMITED,1-STOP CONNECTIONS PTY LIMITED,5.810257e+10,True,<250,7000.0,"Professional, Scientific and Technical Services",Computer System Design and Related Services,Computer System Design and Related Services,Computer System Design and Related Services,Manager,CEOs,Full-time,Permanent,Women,1
1,2024-25,1-STOP CONNECTIONS PTY LIMITED,1-STOP CONNECTIONS PTY LIMITED,5.810257e+10,True,<250,7000.0,"Professional, Scientific and Technical Services",Computer System Design and Related Services,Computer System Design and Related Services,Computer System Design and Related Services,Manager,Other Executives and General Managers,Full-time,Permanent,Men,5
2,2024-25,1-STOP CONNECTIONS PTY LIMITED,1-STOP CONNECTIONS PTY LIMITED,5.810257e+10,True,<250,7000.0,"Professional, Scientific and Technical Services",Computer System Design and Related Services,Computer System Design and Related Services,Computer System Design and Related Services,Manager,Other managers,Full-time,Permanent,Men,15
3,2024-25,1-STOP CONNECTIONS PTY LIMITED,1-STOP CONNECTIONS PTY LIMITED,5.810257e+10,True,<250,7000.0,"Professional, Scientific and Technical Services",Computer System Design and Related Services,Computer System Design and Related Services,Computer System Design and Related Services,Manager,Other managers,Full-time,Permanent,Women,2
4,2024-25,1-STOP CONNECTIONS PTY LIMITED,1-STOP CONNECTIONS PTY LIMITED,5.810257e+10,True,<250,7000.0,"Professional, Scientific and Technical Services",Computer System Design and Related Services,Computer System Design and Related Services,Computer System Design and Related Services,Manager,Senior managers,Full-time,Permanent,Men,3


In [4]:
# Gender / manager_category distribution — sanity check
wc = data["workforce_composition"]
print("Gender values:", wc["gender"].value_counts().to_dict())
print("Manager category values:", wc["manager_category"].value_counts().to_dict())
print("Employer-size bands:", wc["employer_size"].value_counts().to_dict())

Gender values: {'Women': 107077, 'Men': 104582}
Manager category values: {'Non-manager': 141673, 'Manager': 69986}
Employer-size bands: {'<250': 100936, '250-499': 47415, '1000-4999': 28881, '500-999': 27961, '5000+': 6338}


In [5]:
print("workforce_management_statistics columns:")
print(list(data["workforce_management_statistics"].columns))
data["workforce_management_statistics"].head()

workforce_management_statistics columns:
['reporting_year', 'corporate_group_name', 'employer_name', 'employer_abn', 'is_relevant_employer', 'employer_size', 'anzsic_code', 'anzsic_division', 'anzsic_subdivision', 'anzsic_group', 'anzsic_class', 'movement_type', 'manager_category', 'manager_type', 'employment_status', 'employment_type', 'gender', 'n_employees']


,reporting_year,corporate_group_name,employer_name,employer_abn,is_relevant_employer,employer_size,anzsic_code,anzsic_division,anzsic_subdivision,anzsic_group,anzsic_class,movement_type,manager_category,manager_type,employment_status,employment_type,gender,n_employees
0,2024-25,1-STOP CONNECTIONS PTY LIMITED,1-STOP CONNECTIONS PTY LIMITED,5.810257e+10,True,<250,7000.0,"Professional, Scientific and Technical Services",Computer System Design and Related Services,Computer System Design and Related Services,Computer System Design and Related Services,Promotions,Managers,Mid to entry level management,Full-time,Permanent,Women,2
1,2024-25,1-STOP CONNECTIONS PTY LIMITED,1-STOP CONNECTIONS PTY LIMITED,5.810257e+10,True,<250,7000.0,"Professional, Scientific and Technical Services",Computer System Design and Related Services,Computer System Design and Related Services,Computer System Design and Related Services,Promotions,Managers,Mid to entry level management,Full-time,Permanent,Men,3
2,2024-25,1-STOP CONNECTIONS PTY LIMITED,1-STOP CONNECTIONS PTY LIMITED,5.810257e+10,True,<250,7000.0,"Professional, Scientific and Technical Services",Computer System Design and Related Services,Computer System Design and Related Services,Computer System Design and Related Services,Promotions,Non-managers,Non-managers,Full-time,Permanent,Men,1
3,2024-25,1-STOP CONNECTIONS PTY LIMITED,1-STOP CONNECTIONS PTY LIMITED,5.810257e+10,True,<250,7000.0,"Professional, Scientific and Technical Services",Computer System Design and Related Services,Computer System Design and Related Services,Computer System Design and Related Services,Promotions,Managers,Mid to entry level management,Part-time,Permanent,Women,1
4,2024-25,1-STOP CONNECTIONS PTY LIMITED,1-STOP CONNECTIONS PTY LIMITED,5.810257e+10,True,<250,7000.0,"Professional, Scientific and Technical Services",Computer System Design and Related Services,Computer System Design and Related Services,Computer System Design and Related Services,Promtions from non-manager to manager,Managers,Mid to entry level management,Full-time,Permanent,Women,2


## 4. Inspect a questionnaire table (long-format)

Each questionnaire CSV is a **long table**: one row per (employer, question). The policy features we need for RQ1/RQ2 live in the `question_index` / `response` columns. Notebook 02 will pivot these to wide binary flags using the mapping in `src/config.py::QUESTIONNAIRE_FEATURE_MAP`.

In [6]:
qdf = data["questionnaire_workplace_overview"]
print("Columns:", list(qdf.columns))
print("\nUnique question_index values in this file:")
print(qdf["question_index"].value_counts().head(10))
qdf.head()

Columns: ['reporting_year', 'corporate_group_name', 'employer_name', 'employer_abn', 'is_relevant_employer', 'employer_size', 'anzsic_code', 'anzsic_division', 'anzsic_subdivision', 'anzsic_group', 'anzsic_class', 'section', 'subsection', 'question_index', 'upper_question_text', 'question_text', 'additional_question_info_1', 'additional_question_info_2', 'response_type', 'response']

Unique question_index values in this file:
question_index
GB.GB.Ch.M.MCh     8258
GB.GB.Ch.F.FCh     8258
GB.GB.Mem.M.MCh    8258
GB.GB.Mem.F.FCh    8258
WC.OV.Y            7460
WC.GE.Rec.Y        7174
GB.Targ.N          6549
DnI.FPS.Y          6428
WC.GE.Y.PC         6267
GB.LTCM.N          6109
Name: count, dtype: int64


,reporting_year,corporate_group_name,employer_name,employer_abn,is_relevant_employer,employer_size,anzsic_code,anzsic_division,anzsic_subdivision,anzsic_group,anzsic_class,section,subsection,question_index,upper_question_text,question_text,additional_question_info_1,additional_question_info_2,response_type,response
0,2024-25,Legal Vision Pty. Ltd.,LEGALVISION ILP PTY. LTD.,50167804088,True,<250,6931,"Professional, Scientific and Technical Services","Professional, Scientific and Technical Service...",Legal and Accounting Services,Legal Services,Workplace overview,Policy and strategy,DnI.FPS.N,NaN,1.2 Do you have a formal policy and/or formal ...,NaN,NaN,Select,No
1,2024-25,Assetinsure Holdings Pty Limited,Assetinsure Holdings Pty Limited,52103489265,True,<250,6240,Financial and Insurance Services,Finance,Financial Asset Investing,Financial Asset Investing,Workplace overview,Policy and strategy,DnI.FPS.N,NaN,1.2 Do you have a formal policy and/or formal ...,NaN,NaN,Select,No
2,2024-25,Southern Cross Grammar,Southern Cross Grammar,35149437276,True,<250,8023,Education and Training,Preschool and School Education,School Education,Combined Primary and Secondary Education,Workplace overview,Policy and strategy,DnI.FPS.N,NaN,1.2 Do you have a formal policy and/or formal ...,NaN,NaN,Select,No
3,2024-25,Toowoomba Motors Pty Ltd,Toowoomba Motors Pty Ltd,76153536406,True,<250,3911,Retail Trade,Motor Vehicle and Motor Vehicle Parts Retailing,Motor Vehicle Retailing,Car Retailing,Workplace overview,Policy and strategy,DnI.FPS.N,NaN,1.2 Do you have a formal policy and/or formal ...,NaN,NaN,Select,No
4,2024-25,ANZUK Education Services Pty Ltd,ANZUK Education Services Pty Ltd,19123730521,True,<250,7212,Administrative and Support Services,Administrative Services,Employment Services,Labour Supply Services,Workplace overview,Policy and strategy,DnI.FPS.N,NaN,1.2 Do you have a formal policy and/or formal ...,NaN,NaN,Select,No


In [7]:
# Example response distributions for the D&I policy question
dni = qdf[qdf["question_index"] == "DnI.FPS.N"]
print(f"D&I policy (DnI.FPS.N) — {len(dni)} responses")
print(dni["response"].value_counts())

D&I policy (DnI.FPS.N) — 1811 responses
response
No    1811
Name: count, dtype: int64


## 5. Validate — required columns + join-key integrity

In [ ]:
data_acquisition.validate(data)
wc = data["workforce_composition"]
print("Unique employer ABNs (workforce_composition):", wc["employer_abn"].nunique())
print("Reporting years present:", wc["reporting_year"].unique())
print("ANZSIC divisions (head):", wc["anzsic_division"].unique()[:6])

21:39:51 | WARNING | src.data_acquisition | workforce_composition: dropping 20530 rows with null employer_abn (9.70%)
21:39:51 | WARNING | src.data_acquisition | workforce_management_statistics: dropping 21011 rows with null employer_abn (8.91%)
21:39:51 | INFO    | src.data_acquisition | Validation passed: 8239 unique employers in workforce_composition
aaaaaaaaaaaaaaaaaa
{'workforce_composition':        reporting_year            corporate_group_name                   employer_name  employer_abn  is_relevant_employer employer_size  anzsic_code  \
0             2024-25  1-STOP CONNECTIONS PTY LIMITED  1-STOP CONNECTIONS PTY LIMITED  5.810257e+10                  True          <250       7000.0   
1             2024-25  1-STOP CONNECTIONS PTY LIMITED  1-STOP CONNECTIONS PTY LIMITED  5.810257e+10                  True          <250       7000.0   
2             2024-25  1-STOP CONNECTIONS PTY LIMITED  1-STOP CONNECTIONS PTY LIMITED  5.810257e+10                  True          <250       7

## 6. Load optional external heterogeneous source

To score in the higher grade band, Assessment 2 requires integration of at least one additional data source. Drop a CSV at `data/external/abs_gender_pay_gap_by_industry.csv` (columns: `anzsic_division`, `industry_pay_gap`) and it will be picked up here automatically. If absent, the pipeline continues with WGEA alone.

In [9]:
abs_df = data_acquisition.load_external_abs()
if abs_df is not None:
    display(abs_df.head())
else:
    print("No external source loaded — notebook 02 will skip integration.")

21:39:52 | WARNING | src.data_acquisition | External ABS file not found at D:\CDU\Semester3\PRT564_DATA_ANALYTICS_AND_VISUALISATION\project\data\external\abs_gender_pay_gap_by_industry.csv — skipping heterogeneous integration. Place a CSV with columns [anzsic_division, industry_pay_gap] there to enable.
No external source loaded — notebook 02 will skip integration.


## 7. Checkpoint for the next notebook

We pickle the dict of DataFrames + the optional external DataFrame to `data/processed/checkpoints/01_raw_data.pkl`. Notebook 02 loads this rather than re-reading the raw CSVs.

In [10]:
checkpoint = {"data": data, "abs_df": abs_df}
path = save_checkpoint(checkpoint, config.CHECKPOINT_DIR / "01_raw_data.pkl")
print("Saved →", path)
print(f"Size   : {path.stat().st_size / 1e6:.2f} MB")

Saved → D:\CDU\Semester3\PRT564_DATA_ANALYTICS_AND_VISUALISATION\project\data\processed\checkpoints\01_raw_data.pkl
Size   : 216.70 MB


## Summary

- Eight WGEA CSVs loaded and validated.
- Workforce tables are wide (one row per employer × gender × occupation), questionnaires are long (one row per employer × question).
- All required columns present; `employer_abn` populated everywhere.
- Checkpoint written for notebook 02.

**Next:** `02_preprocessing.ipynb` — merge everything into a single employer-level master table.